In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import seaborn as sns


In [ ]:
df = pd.read_csv("data/gst_data.csv")


**Exploratory Data Analysis**



In [ ]:
#Printing the first 5 rows and information
print("First 5 Rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())


In [ ]:
pd.set_option('display.float_format', '{:.4f}'.format)

print("Descriptive Statistics: yearwise")
print(df.describe())


In [ ]:
# Remove the last 4 summary rows
df_filtered = df.iloc[:-3].copy()
years=[]
#separating numeric data from states and removing totals and import data to calculate descriptive statistics
numeric_df = df.select_dtypes(include=np.number)
numeric_df = numeric_df.iloc[:-3]
numeric_df
# Financial year columns
for column in numeric_df.columns:
  years.append(column)


In [ ]:

# Calculating descriptive statistics for each state
state_stats = df.apply(
    lambda row: pd.Series({
        'Mean': row[years].mean(),
        'Median': row[years].median(),
        'Min': row[years].min(),
        'Max': row[years].max(),
        'Std Dev': row[years].std(),
        'Variance': row[years].var()
    }),
    axis=1
)

state_stats.insert(0, 'State/UT', df['State/UT'])

print(state_stats.round(4))


In [ ]:
for column in numeric_df.columns:
    print(f"\n----- {column} -----")
    print("Mean:", numeric_df[column].mean())
    print("Median:", numeric_df[column].median())
    print("Mode:", numeric_df[column].mode().values)
    print("Variance:", numeric_df[column].var())
    print("Standard Deviation:", numeric_df[column].std())
    print("Skewness:", stats.skew(numeric_df[column], nan_policy='omit'))
    print("Kurtosis:", stats.kurtosis(numeric_df[column], nan_policy='omit'))


**Data Visualization**


In [ ]:

# Set State/UT as index for the heatmap matrix
heatmap_data = df_filtered.set_index('State/UT')[years]

# Sort by the latest year to keep top-performing states at the top
heatmap_data = heatmap_data.sort_values(by=years[-1], ascending=False)

# Set figure size to comfortably fit all states and text annotations
plt.figure(figsize=(12, 16))

# Display standard heatmap with numeric values annotated in each cell
sns.heatmap(
    heatmap_data,
    annot=True,              # Display numbers in heatmap cells
    fmt=',.0f',              # Format numbers with comma separators
    cmap='YlGnBu',           # Standard readable sequential color map
    linewidths=0.5,          # Subtle grid lines between cells
    cbar_kws={'label': 'GST Collection (₹ Crore)'}
)

plt.title("State-wise GST Collection Across Years", fontsize=16, pad=15)
plt.xlabel("Financial Year", fontsize=12)
plt.ylabel("State / UT", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:

# Set State/UT as index so matplotlib uses state names along the X-axis
df_plot = df_filtered.set_index('State/UT')[years]

# Plot grouped vertical bar chart
fig, ax = plt.subplots(figsize=(22, 8))
df_plot.plot(kind='bar', width=0.8, ax=ax)

plt.title("State-wise GST Collection (5-Year Comparison)", fontsize=16, pad=15)
plt.xlabel("State / UT", fontsize=12)
plt.ylabel("GST Collection (₹ Crore)", fontsize=12)

# Rotate state tick labels for clarity
plt.xticks(rotation=90, fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Place legend outside the plot area
plt.legend(title="Financial Year", bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:

yearly_means = df_filtered[years].mean()

plt.figure(figsize=(10, 5))
plt.bar(yearly_means.index, yearly_means.values, color='skyblue', edgecolor='black')

plt.title("Mean GST Collection per Year (2020–2025)", fontsize=14)
plt.xlabel("Financial Year", fontsize=12)
plt.ylabel("Mean Value", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:

# Select only Maharashtra and Karnataka
states = ['Maharashtra', 'Karnataka']
df_plot = df[df['State/UT'].isin(states)]

# Data
maharashtra = df_plot[df_plot['State/UT'] == 'Maharashtra'][years].values.flatten()
karnataka = df_plot[df_plot['State/UT'] == 'Karnataka'][years].values.flatten()

# X locations
x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(10,6))

plt.bar(x - width/2, maharashtra, width,
        label='Maharashtra', color='steelblue')

plt.bar(x + width/2, karnataka, width,
        label='Karnataka', color='orange')

plt.xticks(x, years)
plt.xlabel("Financial Year")
plt.ylabel("GST Collection")
plt.title("GST Collection: Maharashtra vs Karnataka")
plt.legend()

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:

# Last row contains Total GST Collection
total = df.iloc[-1]
values = total[years].astype(float)

plt.figure(figsize=(8,8))

plt.pie(values,
        labels=years,
        autopct='%1.1f%%',
        startangle=90,
        shadow=True)

plt.title("Year-wise Total GST Collection")
plt.axis('equal')
plt.show()


In [ ]:
domestic = df.iloc[-3][years].astype(int).tolist()
imports = df.iloc[-2][years].astype(int).tolist()

# Bar positions
x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(10,6))

bars1 = plt.bar(x - width/2, domestic, width,
                label='GST Collection (Domestic)',
                color='royalblue')

bars2 = plt.bar(x + width/2, imports, width,
                label='GST Collection (Imports)',
                color='orange')

# Add value labels
for bar in bars1:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 15000,
             f'{int(bar.get_height()):,}',
             ha='center', fontsize=9)

for bar in bars2:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 15000,
             f'{int(bar.get_height()):,}',
             ha='center', fontsize=9)

plt.xticks(x, years)
plt.xlabel("Financial Year", fontsize=12)
plt.ylabel("GST Collection (₹ Crore)", fontsize=12)
plt.title("Domestic GST Collection vs Import GST Collection", fontsize=14)

plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


**K- Means Clustering**


In [ ]:

# 1. Prepare feature matrix X using the year columns
X = df_filtered[years]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Run K-Means Clustering (e.g., k=3 clusters)
kmeans = KMeans(n_clusters=3, random_state=42)
df_filtered['Cluster'] = kmeans.fit_predict(X_scaled)

# 3. Print states by cluster
df_filtered['Cluster'] = df_filtered['Cluster']  # sync with df_filtered
for i in sorted(df_filtered['Cluster'].unique()):
  print(f'\nCluster {i}')
  print(df_filtered[df_filtered['Cluster'] == i]['State/UT'].tolist())

# 4. Perform PCA & Plot
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 9))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df_filtered['Cluster'],
    cmap='viridis',
    s=100,
)

# Create legend handles
handles = []
for cluster_id in sorted(df_filtered['Cluster'].unique()):
  color = plt.cm.viridis(cluster_id / df_filtered['Cluster'].max())
  handles.append(
      plt.Line2D(
          [0],
          [0],
          marker='o',
          color='w',
          label=f'Cluster {cluster_id}',
          markerfacecolor=color,
          markersize=10,
      )
  )

plt.legend(handles=handles, title='Clusters')

for i, state in enumerate(df_filtered['State/UT']):
  plt.text(X_pca[i, 0] + 0.05, X_pca[i, 1] + 0.05, state, fontsize=8)

plt.title('GST State Clusters (K-Means)', fontsize=16)
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# 1. Calculate YoY growth rate columns (%)
growth_df = pd.DataFrame({'State/UT': df_filtered['State/UT']})

for i in range(1, len(years)):
    prev_yr, curr_yr = years[i - 1], years[i]
    growth_df[f'Growth_{curr_yr}'] = (
        (df_filtered[curr_yr] - df_filtered[prev_yr]) / df_filtered[prev_yr]
    ) * 100

# Feature matrix of growth percentages
X_growth = growth_df.drop(columns=['State/UT'])

# 2. Scale features and run K-Means
scaler = StandardScaler()
X_growth_scaled = scaler.fit_transform(X_growth)

kmeans_growth = KMeans(n_clusters=3, random_state=42)
growth_df['Growth_Cluster'] = kmeans_growth.fit_predict(X_growth_scaled)

# Display results
for cluster_id in sorted(growth_df['Growth_Cluster'].unique()):
    states = growth_df[growth_df['Growth_Cluster'] == cluster_id]['State/UT'].tolist()
    print(f"\n--- Growth Cluster {cluster_id} ---")
    print(states)


In [ ]:

# 1. Reduce the growth rate features (4 YoY growth columns) to 2D using PCA
pca_growth = PCA(n_components=2)
X_growth_pca = pca_growth.fit_transform(X_growth_scaled)

# 2. Plot the growth clusters
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    X_growth_pca[:, 0],
    X_growth_pca[:, 1],
    c=growth_df['Growth_Cluster'],
    cmap='viridis',
    s=100
)

# 3. Create a custom legend
handles = []
for cluster_id in sorted(growth_df['Growth_Cluster'].unique()):
    color = plt.cm.viridis(cluster_id / growth_df['Growth_Cluster'].max())
    handles.append(
        plt.Line2D(
            [0], [0],
            marker='o',
            color='w',
            label=f'Growth Cluster {cluster_id}',
            markerfacecolor=color,
            markersize=10
        )
    )

plt.legend(handles=handles, title='Growth Clusters')

# 4. Add state labels
for i, state in enumerate(growth_df['State/UT']):
    plt.text(X_growth_pca[i, 0] + 0.05, X_growth_pca[i, 1] + 0.05, state, fontsize=8)

plt.title('GST YoY Growth Clustering (K-Means)', fontsize=16)
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:


# Map 3 years to X, Y, Z space, 1 year to marker size, and 1 year to hover details
fig = px.scatter_3d(
    df_filtered,
    x='2020-21',  # Dim 1
    y='2021-22',  # Dim 2
    z='2022-23',  # Dim 3
    color='Cluster',  # Discrete Cluster colors
    size='2024-25',  # Dim 4 (Marker Size)
    hover_name='State/UT',
    hover_data=['2023-24'],  # Dim 5 (Hover details)
    title='5D GST Feature Representation (3D Space + Size + Tooltip)',
    opacity=0.85,
)

fig.update_layout(
    scene=dict(
        xaxis_title='2020-21 GST',
        yaxis_title='2021-22 GST',
        zaxis_title='2022-23 GST',
    ),
    width=900,
    height=700,
)

fig.show()


In [ ]:
# 1. Compute stability metrics across the 5 years
volatility_df = pd.DataFrame({'State/UT': df_filtered['State/UT']})

# Mean, Standard Deviation, and Coefficient of Variation
volatility_df['Mean_GST'] = df_filtered[years].mean(axis=1)
volatility_df['Std_Dev'] = df_filtered[years].std(axis=1)
volatility_df['CV'] = volatility_df['Std_Dev'] / volatility_df['Mean_GST']

# Feature matrix focused on relative dispersion (CV)
X_volatility = volatility_df[['CV']]

# 2. Scale and cluster
X_vol_scaled = scaler.fit_transform(X_volatility)

kmeans_vol = KMeans(n_clusters=3, random_state=42)
volatility_df['Volatility_Cluster'] = kmeans_vol.fit_predict(X_vol_scaled)

# Display results
for cluster_id in sorted(volatility_df['Volatility_Cluster'].unique()):
    states = volatility_df[volatility_df['Volatility_Cluster'] == cluster_id]['State/UT'].tolist()
    print(f"\n--- Volatility Cluster {cluster_id} ---")
    print(states)


In [ ]:
# 1. Scatter Plot of Coefficient of Variation (CV) vs. Mean GST by Volatility Cluster
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=volatility_df,
    x='Mean_GST',
    y='CV',
    hue='Volatility_Cluster',
    palette='viridis',
    s=100
)

# Add state labels for points with higher volatility
for i, row in volatility_df.iterrows():
    if row['CV'] > volatility_df['CV'].median():
        plt.text(row['Mean_GST'] + 500, row['CV'], row['State/UT'], fontsize=8)

plt.title('Volatility Clustering: Coefficient of Variation vs. Mean GST Collection', fontsize=14)
plt.xlabel('Mean GST Collection (₹ Crore)', fontsize=12)
plt.ylabel('Coefficient of Variation (Volatility)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
